# NB3 — YOLOv26 Semantic + Final 3-Model Comparison · LiTS

**Course:** CSE 445 Computer Vision — Assignment Part 1  ·  **NB3 of 4**

Loads the **identical split** from NB0 → converts it to the YOLO semantic format → **Task E** (train ≥50 epochs) → **Task F** (test metrics) → **Task G** (error analysis) → **Task H** (final 3-model comparison, pulling NB1 + NB2 metrics from the shared `results.json`).

- **Model:** Ultralytics `yolo26s-sem.pt` (semantic `-sem`, **not** `-seg`).
- **Mask format YOLO expects:** single-channel PNG, **pixel = class id**, **255 = ignore** — we export our fused `{0,1,2}` labels accordingly.
- **Metrics:** the **same confusion-matrix code** as NB1/NB2 for a fair comparison.

> **Setup on Kaggle:** Accelerator = **GPU T4 x2**, Internet = **ON** (installs ultralytics, downloads the `-sem` checkpoint). **+ Add Input:** LiTS dataset, **NB0 output** (`split.json`), **NB1 output** (`deeplabv3_best.pt`, `results.json`), **NB2 output** (`segformer_best.pt`) — the last two are needed for the Task H side-by-side grid.

In [ ]:
# =====================================================================
# 1. Setup & imports (version pins)  -- ensure ultralytics supports yolo26-sem
# =====================================================================
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "ultralytics"], check=False)

import os, re, json, time, glob, random, shutil, yaml
from pathlib import Path
import numpy as np, pandas as pd, cv2
from PIL import Image
import matplotlib.pyplot as plt
import torch, torch.nn as nn, torch.nn.functional as F
import albumentations as A
from albumentations.pytorch import ToTensorV2
from ultralytics import YOLO
import ultralytics
print("ultralytics", ultralytics.__version__, "| torch", torch.__version__)

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
WORK = Path("/kaggle/working"); WORK.mkdir(parents=True, exist_ok=True)
print("device:", device, "| GPUs:", torch.cuda.device_count())

## 2. Load the shared split from NB0

In [ ]:
# =====================================================================
# 2. Load split.json (from NB0) + rebuild path lookups
# =====================================================================
cand = glob.glob("/kaggle/input/**/split.json", recursive=True)
assert cand, "split.json not found -> + Add Input the NB0-output dataset."
meta = json.load(open(cand[0]))
CLASS_NAMES = meta["class_names"]; NUM_CLASSES = meta["num_classes"]; IMG_SIZE = 256
print("loaded split | seed", meta["seed"], "| filter_to_liver", meta["filter_to_liver"])

IMG_DIR, LIVER_DIR, TUMOR_DIR = (Path(meta["dirs"][k]) for k in ("images","liver","tumor"))
if not IMG_DIR.exists():
    root = Path(next(iter(glob.glob("/kaggle/input/**/Thesis_data", recursive=True))))
    IMG_DIR   = next(p for p in root.iterdir() if p.is_dir() and "image" in p.name.lower())
    LIVER_DIR = next(p for p in root.iterdir() if p.is_dir() and "liver" in p.name.lower())
    TUMOR_DIR = next(p for p in root.iterdir() if p.is_dir() and "tumor" in p.name.lower())

def vskey(p):
    n = [int(x) for x in re.findall(r"\d+", Path(p).stem)]
    return (n[0], n[1]) if len(n) >= 2 else (n[0], -1)
img_by = {vskey(p): p for p in IMG_DIR.glob("*.png")}
liv_by = {vskey(p): p for p in LIVER_DIR.glob("*.png")}
tum_by = {vskey(p): p for p in TUMOR_DIR.glob("*.png")}
split = meta["slices"]
print("slices  train/val/test:", {k: len(v) for k, v in split.items()})

def combined_label(k):
    liv = np.array(Image.open(liv_by[k]).convert("L")) > 0
    tum = np.array(Image.open(tum_by[k]).convert("L")) > 0
    lab = np.zeros((IMG_SIZE, IMG_SIZE), np.uint8); lab[liv]=1; lab[tum]=2
    return lab

## 3. Convert the shared split → YOLO semantic dataset format
YOLO expects parallel `images/{train,val,test}` and `masks/{train,val,test}` folders with **matching stems**, masks single-channel PNG (**pixel = class id**). We symlink the images and generate the fused `{0,1,2}` masks — the split itself is byte-identical to NB1/NB2 (no re-splitting).

In [ ]:
# =====================================================================
# 3. Export images (symlink) + fused masks (PNG 0/1/2) + write data.yaml
# =====================================================================
YROOT = WORK/"yolo_ds"
for sp in ["train","val","test"]:
    (YROOT/"images"/sp).mkdir(parents=True, exist_ok=True)
    (YROOT/"masks"/sp).mkdir(parents=True, exist_ok=True)

t0 = time.time()
for sp in ["train","val","test"]:
    for j, sid in enumerate(split[sp]):
        k = tuple(int(x) for x in sid.split("_"))
        name = img_by[k].name
        dst = YROOT/"images"/sp/name
        if not dst.exists():
            try: os.symlink(img_by[k], dst)
            except Exception: shutil.copy(img_by[k], dst)
        Image.fromarray(combined_label(k)).save(YROOT/"masks"/sp/name)
    print(f"  {sp}: exported {len(split[sp])} pairs ({time.time()-t0:.0f}s)")

data_yaml = {"path": str(YROOT), "train": "images/train", "val": "images/val",
             "test": "images/test", "masks_dir": "masks",
             "names": {i: n for i, n in enumerate(CLASS_NAMES)}}
YAML_PATH = WORK/"lits_sem.yaml"
yaml.safe_dump(data_yaml, open(YAML_PATH, "w"), sort_keys=False)
print("wrote", YAML_PATH); print(open(YAML_PATH).read())

## 4–5. Model + training configuration + Task E (train ≥50 epochs)
Ultralytics manages its own optimizer, LR schedule, loss, and augmentation for the `-sem` task; we report the training arguments we set and rely on the documented defaults for the rest (justified at viva).

In [ ]:
# =====================================================================
# 4-5. CONFIG + train YOLOv26-sem
# =====================================================================
CONFIG = {
    "model": "YOLOv26-sem (Ultralytics)", "checkpoint": "yolo26s-sem.pt",
    "num_classes": NUM_CLASSES, "img_size": IMG_SIZE, "epochs": 50, "batch_size": 16,
    "optimizer": "Ultralytics auto (SGD/AdamW)", "lr": "Ultralytics default (lr0=0.01, cosine)",
    "loss": "Ultralytics semantic loss (built-in)",
    "note": "masks: pixel=class id, 255=ignore; same split as NB0; flips DISABLED (fliplr=flipud=0, lateralized organ)",
}
print(CONFIG)

ymodel = YOLO(CONFIG["checkpoint"])
ymodel.train(data=str(YAML_PATH), epochs=CONFIG["epochs"], imgsz=CONFIG["img_size"],
             batch=CONFIG["batch_size"], seed=SEED, project=str(WORK/"yolo_runs"),
             name="lits_sem", exist_ok=True, verbose=True,
             fliplr=0.0, flipud=0.0)   # medical-grade: no flips (lateralized organ)
try:    BEST = str(ymodel.trainer.best)
except Exception: BEST = str(next(iter((WORK/"yolo_runs").rglob("best.pt"))))
print("best weights:", BEST)

In [ ]:
# per-epoch curves from Ultralytics results.csv (Task E requirement)
csvs = list((WORK/"yolo_runs").rglob("results.csv"))
if csvs:
    df = pd.read_csv(csvs[0]); df.columns = [c.strip() for c in df.columns]
    loss_cols = [c for c in df.columns if "loss" in c.lower()]
    miou_cols = [c for c in df.columns if "miou" in c.lower() or ("iou" in c.lower() and "metric" in c.lower())]
    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    for c in loss_cols: ax[0].plot(df[c], label=c)
    ax[0].set_title("YOLO losses"); ax[0].set_xlabel("epoch"); ax[0].legend(fontsize=6)
    for c in (miou_cols or [c for c in df.columns if "metric" in c.lower()]): ax[1].plot(df[c], label=c)
    ax[1].set_title("YOLO val metrics"); ax[1].set_xlabel("epoch"); ax[1].legend(fontsize=6)
    plt.tight_layout(); plt.savefig(WORK/"yolo_curves.png", dpi=120); plt.show()
else:
    print("results.csv not found under", WORK/"yolo_runs")

## 6. Loss-free metrics (Task F) — same confusion-matrix code as NB1/NB2

In [ ]:
# =====================================================================
# 6. Confusion-matrix metrics (identical to NB1/NB2)
# =====================================================================
class ConfMat:
    def __init__(self, n): self.n = n; self.mat = torch.zeros(n, n, dtype=torch.int64)
    def update(self, pred, tgt):
        k = (tgt >= 0) & (tgt < self.n)
        idx = self.n * tgt[k].to(torch.int64) + pred[k].to(torch.int64)
        self.mat += torch.bincount(idx, minlength=self.n**2).reshape(self.n, self.n)
    def compute(self):
        m = self.mat.double(); tp = m.diag(); fp = m.sum(0)-tp; fn = m.sum(1)-tp
        iou = tp/(tp+fp+fn).clamp(min=1e-9); dice = 2*tp/(2*tp+fp+fn).clamp(min=1e-9)
        recall = tp/m.sum(1).clamp(min=1e-9)
        return {"iou": iou.tolist(), "miou": iou.mean().item(), "dice": dice.tolist(),
                "mdice": dice.mean().item(), "pixel_acc": (tp.sum()/m.sum().clamp(min=1e-9)).item(),
                "mean_pixel_acc": recall.mean().item()}

def image_iou(pred, tgt):
    ious = []
    for c in range(NUM_CLASSES):
        ti = tgt == c
        if ti.sum() == 0: continue
        pi = pred == c
        inter = (pi & ti).sum(); union = (pi | ti).sum()
        ious.append(inter/union if union else 0.0)
    return float(np.mean(ious)) if ious else 1.0

def yolo_pred(model, k):
    r = model.predict(str(img_by[k]), imgsz=IMG_SIZE, verbose=False)[0]
    d = r.semantic_mask.data
    d = d.cpu().numpy() if hasattr(d, "cpu") else np.array(d)
    d = np.squeeze(d)
    if d.shape != (IMG_SIZE, IMG_SIZE):
        d = cv2.resize(d.astype(np.uint8), (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_NEAREST)
    return d.astype(np.uint8)
print("metrics ready")

In [ ]:
# =====================================================================
# 6b. Task F — predict on held-out TEST split, accumulate metrics
# =====================================================================
from collections import defaultdict
best_model = YOLO(BEST)
cmref = ConfMat(NUM_CLASSES); vol_cms = defaultdict(lambda: ConfMat(NUM_CLASSES)); per_img = []
t0 = time.time()
for j, sid in enumerate(split["test"]):
    k = tuple(int(x) for x in sid.split("_"))
    pred = yolo_pred(best_model, k); gt = combined_label(k)
    pt, gtt = torch.from_numpy(pred), torch.from_numpy(gt)
    cmref.update(pt, gtt); vol_cms[k[0]].update(pt, gtt)
    per_img.append((image_iou(pred, gt), sid))
    if (j+1) % 500 == 0: print(f"  {j+1}/{len(split['test'])} ({time.time()-t0:.0f}s)")
testm = cmref.compute()

def _dice_c(cm, c):
    m=cm.mat.double(); tp=m[c,c]; fp=m[:,c].sum()-tp; fn=m[c,:].sum()-tp
    return (2*tp/(2*tp+fp+fn).clamp(min=1e-9)).item()
def _sens_c(cm, c):
    m=cm.mat.double(); tp=m[c,c]; fn=m[c,:].sum()-tp
    return (tp/(tp+fn).clamp(min=1e-9)).item()
tumor_sens=_sens_c(cmref,2); liver_sens=_sens_c(cmref,1)
liver_vd=[_dice_c(cm,1) for cm in vol_cms.values() if cm.mat[1,:].sum()>0]
tumor_vd=[_dice_c(cm,2) for cm in vol_cms.values() if cm.mat[2,:].sum()>0]
pv_liver=(float(np.mean(liver_vd)), float(np.std(liver_vd)))
pv_tumor=(float(np.mean(tumor_vd)), float(np.std(tumor_vd)))

print("=== YOLOv26-sem — TEST metrics ===")
print(pd.DataFrame({"class": CLASS_NAMES, "IoU": np.round(testm["iou"],4),
                    "Dice": np.round(testm["dice"],4)}).to_string(index=False))
print(f"\nmIoU {testm['miou']:.4f} | mean Dice {testm['mdice']:.4f} | "
      f"pixel acc {testm['pixel_acc']:.4f} | mean pixel acc {testm['mean_pixel_acc']:.4f}")
print("\n--- CLINICAL METRICS (medical-grade) ---")
print(f"tumor sensitivity (recall) : {tumor_sens:.4f}   (miss rate {1-tumor_sens:.1%})")
print(f"liver sensitivity (recall) : {liver_sens:.4f}")
print(f"per-patient liver Dice     : {pv_liver[0]:.4f} +/- {pv_liver[1]:.4f}  (n={len(liver_vd)})")
print(f"per-patient tumor Dice     : {pv_tumor[0]:.4f} +/- {pv_tumor[1]:.4f}  (n={len(tumor_vd)})")

cmn = cmref.mat.double(); cmn = (cmn/cmn.sum(1, keepdim=True).clamp(min=1e-9)).numpy()
plt.figure(figsize=(5,4)); plt.imshow(cmn, cmap="Blues", vmin=0, vmax=1)
plt.xticks(range(NUM_CLASSES), CLASS_NAMES, rotation=45); plt.yticks(range(NUM_CLASSES), CLASS_NAMES)
plt.xlabel("predicted"); plt.ylabel("true"); plt.title("YOLOv26-sem pixel confusion (row-norm)")
for i in range(NUM_CLASSES):
    for jj in range(NUM_CLASSES): plt.text(jj, i, f"{cmn[i,jj]:.2f}", ha="center", va="center", fontsize=8)
plt.colorbar(); plt.tight_layout(); plt.savefig(WORK/"yolo_confusion.png", dpi=120); plt.show()

## 7. Error analysis (Task G)
Worst test slices by per-image IoU + most-confused class pair.

**Hypotheses:** tumor is the hardest class (~0.4% of pixels, tiny lesions); YOLO's grid/anchor lineage and downsampling may miss the smallest tumors, so **tumor<->liver** confusion is expected to dominate — quantified below.

In [ ]:
# =====================================================================
# 7. Task G — worst-5 test slices (pred vs GT) + most-confused pair
# =====================================================================
per_img.sort(key=lambda t: t[0]); worst = per_img[:5]
cmap = plt.matplotlib.colors.ListedColormap(["black","gold","red"])
fig, ax = plt.subplots(len(worst), 3, figsize=(9, 3*len(worst)))
for i, (sc, sid) in enumerate(worst):
    k = tuple(int(x) for x in sid.split("_"))
    im = np.array(Image.open(img_by[k]).convert("L")); gt = combined_label(k); pr = yolo_pred(best_model, k)
    ax[i,0].imshow(im, cmap="gray"); ax[i,0].set_title(sid, fontsize=8)
    ax[i,1].imshow(gt, cmap=cmap, vmin=0, vmax=2); ax[i,1].set_title("ground truth", fontsize=8)
    ax[i,2].imshow(pr, cmap=cmap, vmin=0, vmax=2); ax[i,2].set_title(f"pred (IoU={sc:.2f})", fontsize=8)
    for j in range(3): ax[i,j].axis("off")
plt.suptitle("YOLOv26-sem — worst-5 test slices", y=1.001)
plt.tight_layout(); plt.savefig(WORK/"yolo_worst.png", dpi=120); plt.show()

off = cmref.mat.clone(); off.fill_diagonal_(0)
a, b = np.unravel_index(off.numpy().argmax(), off.shape)
print(f"most-confused class pair: TRUE '{CLASS_NAMES[a]}' -> PRED '{CLASS_NAMES[b]}' ({off[a,b].item()} px)")

In [ ]:
# =====================================================================
# 7b. Append YOLO metrics to the shared results.json
# =====================================================================
res_path = WORK/"results.json"
src = next(iter(glob.glob("/kaggle/input/**/results.json", recursive=True)), None)
res = json.load(open(src)) if src else (json.load(open(res_path)) if res_path.exists() else {})
res["YOLOv26-sem"] = {
    "miou": testm["miou"], "iou_per_class": testm["iou"], "dice": testm["mdice"],
    "dice_per_class": testm["dice"], "pixel_acc": testm["pixel_acc"],
    "mean_pixel_acc": testm["mean_pixel_acc"],
    "tumor_sensitivity": tumor_sens, "liver_sensitivity": liver_sens,
    "pv_liver_dice": pv_liver, "pv_tumor_dice": pv_tumor,
    "class_names": CLASS_NAMES, "config": CONFIG,
}
json.dump(res, open(res_path, "w"), indent=2)
print("results.json now has:", list(res.keys()))

## 8. Task H — Final 3-Model Comparison
Summary table, mIoU/Dice bar chart, side-by-side qualitative grid (same test slices, all three models + GT), and a written verdict. DeepLabV3 and SegFormer predictions are produced by loading their best checkpoints from NB1/NB2 outputs.

In [ ]:
# =====================================================================
# 8a. Summary table (all models, all Task F metrics)
# =====================================================================
rows = []
for name, r in res.items():
    rows.append({"model": name, "mIoU": round(r["miou"],4), "mean_Dice": round(r["dice"],4),
                 "tumor_sens": round(r.get("tumor_sensitivity", float("nan")),3),
                 "pv_tumor_Dice": round((r.get("pv_tumor_dice") or [float("nan")])[0],3),
                 "pixel_acc": round(r["pixel_acc"],4), "mean_pixel_acc": round(r["mean_pixel_acc"],4),
                 "IoU_liver": round(r["iou_per_class"][1],3), "IoU_tumor": round(r["iou_per_class"][2],3)})
summary = pd.DataFrame(rows).sort_values("mIoU", ascending=False).reset_index(drop=True)
summary.to_csv(WORK/"task_h_summary.csv", index=False)
display(summary)

In [ ]:
# =====================================================================
# 8b. Bar chart — mIoU + mean Dice across the three models
# =====================================================================
models = summary["model"].tolist(); x = np.arange(len(models)); w = 0.35
plt.figure(figsize=(7,4))
plt.bar(x-w/2, summary["mIoU"], w, label="mIoU")
plt.bar(x+w/2, summary["mean_Dice"], w, label="mean Dice")
plt.xticks(x, models, rotation=15); plt.ylim(0,1); plt.ylabel("score"); plt.legend()
plt.title("Task H — model comparison (mIoU & mean Dice)")
for i,v in enumerate(summary["mIoU"]): plt.text(x[i]-w/2, v, f"{v:.2f}", ha="center", va="bottom", fontsize=8)
plt.tight_layout(); plt.savefig(WORK/"task_h_bars.png", dpi=120); plt.show()

In [ ]:
# =====================================================================
# 8c. Side-by-side qualitative grid: same test slices, all 3 models + GT
# =====================================================================
norm = A.Compose([A.Normalize((0.485,0.456,0.406),(0.229,0.224,0.225)), ToTensorV2()])
def prep(k): return norm(image=np.array(Image.open(img_by[k]).convert("RGB")))["image"].unsqueeze(0).to(device)

def load_deeplab():
    p = next(iter(glob.glob("/kaggle/input/**/deeplabv3_best.pt", recursive=True)), None)
    if not p: return None
    from torchvision.models.segmentation import deeplabv3_resnet50
    m = deeplabv3_resnet50(weights=None, aux_loss=True)
    m.classifier[-1] = nn.Conv2d(256, NUM_CLASSES, 1); m.aux_classifier[-1] = nn.Conv2d(256, NUM_CLASSES, 1)
    m.load_state_dict(torch.load(p, map_location=device)); return m.to(device).eval()
def load_segformer():
    p = next(iter(glob.glob("/kaggle/input/**/segformer_best.pt", recursive=True)), None)
    if not p: return None
    from transformers import SegformerForSemanticSegmentation
    m = SegformerForSemanticSegmentation.from_pretrained(
        "nvidia/segformer-b0-finetuned-ade-512-512", num_labels=NUM_CLASSES, ignore_mismatched_sizes=True)
    m.load_state_dict(torch.load(p, map_location=device)); return m.to(device).eval()

dl, sf = load_deeplab(), load_segformer()
def pred_dl(k):
    with torch.no_grad(): return dl(prep(k))["out"].argmax(1)[0].cpu().numpy()
def pred_sf(k):
    with torch.no_grad():
        lo = sf(pixel_values=prep(k)).logits
        return F.interpolate(lo, (IMG_SIZE,IMG_SIZE), mode="bilinear", align_corners=False).argmax(1)[0].cpu().numpy()

# pick 5 tumor-bearing test slices for a meaningful comparison
tum_sids = [s for s in split["test"] if combined_label(tuple(int(x) for x in s.split("_")))[...,].max()==2][:5]
show = tum_sids if len(tum_sids) >= 5 else split["test"][:5]
cols = ["image","GT","DeepLabV3","SegFormer-B0","YOLOv26-sem"]
fig, ax = plt.subplots(len(show), 5, figsize=(15, 3*len(show)))
for i, sid in enumerate(show):
    k = tuple(int(x) for x in sid.split("_"))
    im = np.array(Image.open(img_by[k]).convert("L")); gt = combined_label(k)
    panels = [im, gt,
              pred_dl(k) if dl else np.zeros_like(gt),
              pred_sf(k) if sf else np.zeros_like(gt),
              yolo_pred(best_model, k)]
    for j, (title, pan) in enumerate(zip(cols, panels)):
        if j == 0: ax[i,j].imshow(pan, cmap="gray")
        else: ax[i,j].imshow(pan, cmap=cmap, vmin=0, vmax=2)
        if i == 0: ax[i,j].set_title(title, fontsize=9)
        ax[i,j].axis("off")
    ax[i,0].set_ylabel(sid, fontsize=8)
plt.suptitle("Task H — same test slices across all three models (liver=gold, tumor=red)", y=1.001)
plt.tight_layout(); plt.savefig(WORK/"task_h_qualitative.png", dpi=120); plt.show()
if dl is None or sf is None:
    print("NOTE: attach NB1/NB2 outputs (deeplabv3_best.pt / segformer_best.pt) for their columns.")

In [ ]:
# =====================================================================
# 8d. Written verdict (data-driven)
# =====================================================================
from IPython.display import Markdown, display
best_row = summary.iloc[0]
hardest = min([("liver", summary["IoU_liver"].max()), ("tumor", summary["IoU_tumor"].max())], key=lambda t: t[1])[0]
display(Markdown(f"""### Task H — Verdict
- **Best mIoU:** **{best_row['model']}** at mIoU **{best_row['mIoU']:.3f}** (mean Dice {best_row['mean_Dice']:.3f}).
- **Hardest class across all models:** **{hardest}** — consistent with its tiny pixel share and small, low-contrast lesions.
- **Efficiency:** SegFormer-B0 (~3.7M params) and YOLOv26-sem (small) are lighter than DeepLabV3-ResNet50 (~42M); check the `train_time_min` per model in `results.json` for the speed/accuracy trade-off.
- **Deployment recommendation:** pick the model with the best **tumor IoU** at acceptable latency — for clinical liver-lesion screening, tumor recall matters more than background accuracy. Fill in the concrete numbers from the table above at viva.

*See `task_h_summary.csv`, `task_h_bars.png`, and `task_h_qualitative.png`.*"""))

## References (Academic Integrity)
- Ultralytics, *YOLO Semantic Segmentation* — https://docs.ultralytics.com/tasks/semantic/ and *Dataset format* — https://docs.ultralytics.com/datasets/semantic/
- PyTorch/torchvision *DeepLabV3*; HuggingFace *SegFormer*; Albumentations docs (cited in NB1/NB2).
All pipeline/metric code is our own; official documentation is cited where its APIs were used.